## Setup
Required packages: numpy, os, tifffile, cellpose, matplotlib. 
Cellpose has some strange dependecy requirements; highly recommend setting up new conda environment (conda create env -n cellpose) before pip installing.

Goals: Use manually segmented masks to optimize parameters, then run on whole dataset.

In [1]:
#if set up env called cellpose, can activate here
#%conda init
#%conda activate cellpose

In [ ]:
#install all dependencies - RUN THIS CELL ONCE
#if doesn't work, try installing in console; just remove %

%pip install "cellpose[all]"
%pip install matplotlib

Note: you may need to restart the kernel to use updated packages.
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   ---------------------------------------- 0

In [1]:
#imports 
import numpy as np
import os
import tifffile
from cellpose import models, io, plot, utils, metrics
import matplotlib.pyplot as plt

## Ground Truth Predictions

First, manually segment masks, either through cellpose GUI (recommended for ease of access) OR ImageJ/napari/other image processing software. Manually segmented masks should be stored in their own directory. 

In [25]:
# INPUT DIRECTORIES
# Also, write indices of ground truth data
raw_dir = 'phase-contrast_data'
manual_dir = 'manual_masks'
target_indices = ['00001', '00002', '00010', 
                  '00011', '00012', '00016', 
                  '00025', '00030']

In [17]:
#initialize model. Change model as needed. "bact_phase_cp" works well
#for bacterial phase contrast images
model = models.CellposeModel(gpu=True, 
                             model_type='bact_phase_cp') 

model_type argument is not used in v4.0.1+. Ignoring this argument...


In [19]:
masks_gt = []
masks_pred = []

#Loop through indices of manually labelled 
for idx in target_indices:
    #load image using tifffile.imread() 
    raw_path = os.path.join(raw_dir, f"Phase_Hn249_{idx}.tif")
    image = tifffile.imread(raw_path)
    
    # Assuming using cellpose, which saves files as _seg.npy.
    # otherwise, need to convert to appropriate form (see documentation)
    # error check to ensure file exists, in case of mislabeling.
    npy_path = os.path.join(manual_dir, f"Phase_Hn249_{idx}_seg.npy")
    if os.path.exists(npy_path):
        gt_data = np.load(npy_path, allow_pickle=True).item()
        gt_mask = gt_data['masks']
        masks_gt.append(gt_mask)
    else:
        print(f"Warning: {npy_path} not found.")
        continue

    # run prediction using default parameters; tweak if necessary. 
    pred_mask, flows, styles = model.eval(image, 
                                          diameter=None, 
                                          channels=[0,0])
    masks_pred.append(pred_mask)


channels deprecated in v4.0.1+. If data contain more than 3 channels, only the first 3 channels will be used
channels deprecated in v4.0.1+. If data contain more than 3 channels, only the first 3 channels will be used
channels deprecated in v4.0.1+. If data contain more than 3 channels, only the first 3 channels will be used
channels deprecated in v4.0.1+. If data contain more than 3 channels, only the first 3 channels will be used
channels deprecated in v4.0.1+. If data contain more than 3 channels, only the first 3 channels will be used
channels deprecated in v4.0.1+. If data contain more than 3 channels, only the first 3 channels will be used
channels deprecated in v4.0.1+. If data contain more than 3 channels, only the first 3 channels will be used
channels deprecated in v4.0.1+. If data contain more than 3 channels, only the first 3 channels will be used


## Metrics of Ground Truth Predictions
To evaluate the efficacy of the model, we use a mAP at several IoU thresholds (0.5, 0.75 and 0.85). 

In [23]:
#IoU thresholds as defined 
thresholds = [0.5, 0.75, 0.9]
#mAP at IoU thresholds above, as calculated by cellpose.metrics
ap, tp, fp, fn = metrics.average_precision(masks_gt, 
                                           masks_pred, 
                                           threshold=thresholds)

# ap shape is (n_images, n_thresholds). Calculating the mean...
mean_ap = ap.mean(axis=0) 

print("AGGREGATE PERFORMANCE METRICS")
#printing the output of the metrics for 
for i, t in enumerate(thresholds):
    print(f"mAP @ IoU {t}: {mean_ap[i]:.3f}")

#counting uniquely labeled cells
gt_counts = [len(np.unique(m)) - 1 for m in masks_gt]
pred_counts = [len(np.unique(m)) - 1 for m in masks_pred]

print(f"DATA ANALYSIS SUMMARY")
print(f"Total Bacteria (ground truth): {sum(gt_counts)}")
print(f"Total Bacteria (Cellpose): {sum(pred_counts)}")

# calculating average error per image. 
# this number should be as close to 0 as possible
avg_error = np.mean([abs(g - p) for g, p in zip(gt_counts, pred_counts)])
print(f"Average Error per Image: {avg_error:.1f} cells")

AGGREGATE PERFORMANCE METRICS
mAP @ IoU 0.5: 0.844
mAP @ IoU 0.75: 0.844
mAP @ IoU 0.9: 0.144
DATA ANALYSIS SUMMARY
Total Bacteria (ground truth): 34
Total Bacteria (Cellpose): 32
Average Error per Image: 0.2 cells


## Run model on rest of data

Once happy with parameters (as set above), run the model on the rest of the dataset. The GT analyses can be extrapolated to rest of dataset provided visual inspection is done (see book 02_cellpose_visualization)

In [24]:
#paths
input_dir = './phase-contrast_data'
output_dir = './cellpose_masks'
#if need to create dir, then run below
os.makedirs(output_dir, exist_ok=True)

#check to see input/output dir as expected
#os.listdir(input_dir)
#files = [f for f in os.listdir(input_dir) if f.endswith('.tif')]
#print(files)
#os.listdir(output_dir)


In [10]:
# 2. Initialize the model specifically for bacterial phase contrast
# Using 'bact_phase_cp' (Cellpose-style)
model = models.CellposeModel(gpu=False,
                             model_type='bact_phase_cp')

model_type argument is not used in v4.0.1+. Ignoring this argument...


In [ ]:
files = [f for f in os.listdir(input_dir) if f.endswith('.tif')]

for filename in files:
    print(f"processing {filename}")
    img_path = os.path.join(input_dir, filename)
    image = tifffile.imread(img_path)

    #image should be 2D. If not, will throw error
    if image.ndim > 2:
        image = image[0] 

    #run segmentation
    masks, flows, styles = model.eval(
                                    image, 
                                    diameter=None, 
                                    channels=[0,0],
                                    flow_threshold=0.4,
                                    cellprob_threshold=0.0,
                                    rescale=None 
    )

    # save output; masks should be uint16 so they can be opened in 
    # ImageJ easily (as well as for other SMLM steps)
    mask_filename = os.path.join(output_dir, f"mask_{filename}")
    tifffile.imwrite(mask_filename, masks.astype(np.uint16))
    
    print(f"{filename} processed, found {masks.max()}# of cells")

channels deprecated in v4.0.1+. If data contain more than 3 channels, only the first 3 channels will be used
channels deprecated in v4.0.1+. If data contain more than 3 channels, only the first 3 channels will be used


Processed bacterial image: Phase_Hn249_00001.tif - Found 3 cells


channels deprecated in v4.0.1+. If data contain more than 3 channels, only the first 3 channels will be used


Processed bacterial image: Phase_Hn249_00002.tif - Found 4 cells


channels deprecated in v4.0.1+. If data contain more than 3 channels, only the first 3 channels will be used


...

Processed bacterial image: Phase_Hn249_00032.tif - Found 3 cells


channels deprecated in v4.0.1+. If data contain more than 3 channels, only the first 3 channels will be used


Processed bacterial image: Phase_Hn249_00033.tif - Found 2 cells


channels deprecated in v4.0.1+. If data contain more than 3 channels, only the first 3 channels will be used


Processed bacterial image: Phase_Hn249_00034.tif - Found 2 cells
Processed bacterial image: Phase_Hn249_00035.tif - Found 2 cells
